In [ ]:
import pandas as pd
import osmnx as ox
import folium
from rapidfuzz import process
from geopy.distance import geodesic

# =====================================================
# CONFIGURACIÓN GENERAL
# =====================================================

DISTANCIA_MAXIMA = 500   # metros permitidos desde la SED
SCORE_MINIMO = 80        # confianza mínima de coincidencia
DISTANCIA_DESCARGA = 400 # radio descarga red vial

# =====================================================
# CARGAR BASE DE DATOS SED
# =====================================================

ruta = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\PROYECTO GEOLOCALIZACION SED\IMPUT\DATA SED.xlsx"

df = pd.read_excel(ruta)
df.columns = df.columns.str.strip()

df_sed = df[['SED', 'LONGITUD\nX', 'LATITUD\nY']].dropna().copy()

# =====================================================
# SIMULACIÓN DE FALLAS (PRUEBA)
# =====================================================

datos_fallas = [
    {"SED": df_sed['SED'].iloc[0], "CALLE1": "Calle 18", "CALLE2": "Calle 10"},
    {"SED": df_sed['SED'].iloc[1], "CALLE1": "San Martin", "CALLE2": "Av Lima"},
]

# =====================================================
# PROCESAMIENTO PRINCIPAL
# =====================================================
# ==========================================
# CREAR MAPA GLOBAL (ANTES DEL LOOP)
# ==========================================

mapa_global = folium.Map(
    location=[
        df_sed['LATITUD\nY'].mean(),
        df_sed['LONGITUD\nX'].mean()
    ],
    zoom_start=13
)

for falla in datos_fallas:

    sed_codigo = falla["SED"]
    calle1_input = falla["CALLE1"]
    calle2_input = falla["CALLE2"]

    print("\n====================================")
    print(f"Procesando SED: {sed_codigo}")

    sed_info = df_sed[df_sed['SED'] == sed_codigo]

    if sed_info.empty:
        print("❌ SED no encontrada")
        continue

    lat = sed_info['LATITUD\nY'].values[0]
    lon = sed_info['LONGITUD\nX'].values[0]

    # =====================================================
    # DESCARGAR RED VIAL
    # =====================================================

    G = ox.graph_from_point((lat, lon), dist=DISTANCIA_DESCARGA, network_type='drive')
    nodes, edges = ox.graph_to_gdfs(G)
    edges = edges.reset_index()

    # Limpiar nombres
    edges = edges[edges['name'].notna()].copy()
    edges['name'] = edges['name'].apply(
        lambda x: x[0] if isinstance(x, list) else x
    )

    lista_calles = edges['name'].unique()

    # =====================================================
    # MATCHING CON UMBRAL
    # =====================================================

    match1 = process.extractOne(calle1_input, lista_calles)
    match2 = process.extractOne(calle2_input, lista_calles)

    print("Match 1:", match1)
    print("Match 2:", match2)

    if match1 is None or match1[1] < SCORE_MINIMO:
        print("❌ Calle 1 con baja confianza → REVISIÓN")
        continue

    if match2 is None or match2[1] < SCORE_MINIMO:
        print("❌ Calle 2 con baja confianza → REVISIÓN")
        continue

    calle1_real = match1[0]
    calle2_real = match2[0]

    edges_calle1 = edges[edges['name'] == calle1_real]
    edges_calle2 = edges[edges['name'] == calle2_real]

    if edges_calle1.empty or edges_calle2.empty:
        print("❌ No se encontraron calles en la red")
        continue

    nodos_calle1 = set(edges_calle1['u']).union(set(edges_calle1['v']))
    nodos_calle2 = set(edges_calle2['u']).union(set(edges_calle2['v']))

    nodo_interseccion = nodos_calle1.intersection(nodos_calle2)

    punto_falla = None

    # =====================================================
    # CASO 3: ENTRE CALLES (múltiples intersecciones)
    # =====================================================

    if len(nodo_interseccion) > 1:

        print("🔎 Caso: Entre calles")

        nodos_lista = list(nodo_interseccion)
        nodos_con_distancia = []

        for n in nodos_lista:
            lat_n = nodes.loc[n]['y']
            lon_n = nodes.loc[n]['x']
            dist = geodesic((lat, lon), (lat_n, lon_n)).meters
            nodos_con_distancia.append((n, dist))

        nodos_con_distancia.sort(key=lambda x: x[1])

        nodo1 = nodos_con_distancia[0][0]
        nodo2 = nodos_con_distancia[1][0]

        lat1 = nodes.loc[nodo1]['y']
        lon1 = nodes.loc[nodo1]['x']

        lat2 = nodes.loc[nodo2]['y']
        lon2 = nodes.loc[nodo2]['x']

        lat_mid = (lat1 + lat2) / 2
        lon_mid = (lon1 + lon2) / 2

        punto_falla = (lat_mid, lon_mid)

        print("✅ Punto entre intersecciones cercanas calculado")

    # =====================================================
    # CASO 1: INTERSECCIÓN EXACTA
    # =====================================================

    elif len(nodo_interseccion) == 1:

        print("🔎 Caso: Intersección exacta")

        nodo = list(nodo_interseccion)[0]
        lat_inter = nodes.loc[nodo]['y']
        lon_inter = nodes.loc[nodo]['x']

        punto_falla = (lat_inter, lon_inter)

        print("✅ Intersección encontrada")

    # =====================================================
    # CASO 2: MITAD DE TRAMO MÁS CERCANO
    # =====================================================

    else:

        print("🔎 Caso: Mitad de tramo cercano")

        avenida_union = edges_calle2.unary_union

        distancia_min = float("inf")
        segmento_correcto = None

        for idx, row in edges_calle1.iterrows():
            distancia_geom = row.geometry.distance(avenida_union)
            if distancia_geom < distancia_min:
                distancia_min = distancia_geom
                segmento_correcto = row.geometry

        if segmento_correcto is None:
            print("❌ No se pudo determinar segmento → REVISIÓN")
            continue

        punto_geom = segmento_correcto.interpolate(0.5, normalized=True)
        punto_falla = (punto_geom.y, punto_geom.x)

        print("✅ Punto medio del tramo seleccionado")

    # =====================================================
    # VALIDAR DISTANCIA A LA SED
    # =====================================================

    distancia_final = geodesic((lat, lon), punto_falla).meters
    print("Distancia a la SED:", round(distancia_final, 2), "m")

    if distancia_final > DISTANCIA_MAXIMA:
        print("⚠ Punto demasiado lejano → REVISIÓN")
        continue

    # =====================================================
# ==========================================
# AGREGAR AL MAPA GLOBAL
# ==========================================

folium.Marker(
    [lat, lon],
    popup=f"SED: {sed_codigo}",
    icon=folium.Icon(color="blue")
).add_to(mapa_global)

folium.Marker(
    [punto_falla[0], punto_falla[1]],
    popup=f"Falla - SED {sed_codigo}",
    icon=folium.Icon(color="red")
).add_to(mapa_global)

print("✔ Falla agregada al mapa global")
# ==========================================
# GUARDAR MAPA GLOBAL
# ==========================================

mapa_global.save("REPORTE_FALLAS_DIA.html")

print("\n🔥 REPORTE GLOBAL GENERADO: REPORTE_FALLAS_DIA.html")




Procesando SED: 00001S
Match 1: ('Calle 18', 100.0, 20)
Match 2: ('Calle 10', 100.0, 21)
🔎 Caso: Intersección exacta
✅ Intersección encontrada
Distancia a la SED: 118.3 m
✅ Mapa generado: mapa_00001S.html

Procesando SED: 00002S
Match 1: ('San Francisco Solano', 85.5, 25)
Match 2: ('Avenida Prolongación Tacna', 51.42857142857142, 9)
❌ Calle 2 con baja confianza → REVISIÓN
